### 2023, 2024 복숭아심식나방 데이터 추출

In [ ]:
# -*- coding: utf-8 -*-
import os, re, glob
import pandas as pd

# ================== 설정 ==================
INPUT_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/관찰데이터+지점ID+datetime"
OUT_DIR   = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방"
os.makedirs(OUT_DIR, exist_ok=True)

# 병해(해충) 이름/키워드
PEST_NAME = "복숭아심식나방"
PEST_KEYS = ["복숭아심식나방"]  # 이 문자열이 포함된 컬럼을 모두 병해 관련으로 선택

# 탐색 방식
RECURSIVE = False  # 하위 폴더까지 포함하려면 True

# 연도 필터
YEAR_START = 2023
YEAR_END   = 2024

# ▶ 표준(Core) 스키마: 모델/조인에 공통으로 쓸 컬럼
CORE = [
    "지점ID", "조사년도", "지역-시도", "지역-시군구", "지역-읍면동",
    "상세주소", "좌표-경도", "좌표-위도", "조사회차",
    "조사일자(YYYYMMDD)",  # ← 뒤에 작물명
    "작물명", "품종명"
]

# ▶ Extras 보존 여부/방식
KEEP_EXTRAS = False        # False면 extras 완전 제거
PREFIX_EXTRAS = True       # True면 'CROP__컬럼명' 접두어로 이름 충돌 방지


def detect_crop_from_name(fname: str) -> str:
    """
    파일명에서 작물명 추정. (예: RDA_OBSERVATION_PEACH_with_siteid_dt.csv -> PEACH)
    """
    m = re.search(r"RDA_OBSERVATION_([^_]+)_with_siteid_dt\.csv", fname)
    return m.group(1).upper() if m else "UNKNOWN"


def load_one(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")

    # 작물명 보정(없으면 파일명에서 추정)
    if "작물명" not in df.columns:
        df["작물명"] = detect_crop_from_name(os.path.basename(path))

    # 날짜 파싱
    if "조사일자(YYYYMMDD)" in df.columns:
        # 숫자/문자 혼재해도 안전하게 파싱
        df["조사일자(YYYYMMDD)"] = pd.to_datetime(df["조사일자(YYYYMMDD)"], errors="coerce")
        # 2023~2024 필터
        df = df[df["조사일자(YYYYMMDD)"].dt.year.between(YEAR_START, YEAR_END)]
        # 조사년도 생성/보정
        if "조사년도" not in df.columns:
            df["조사년도"] = df["조사일자(YYYYMMDD)"].dt.year
        else:
            # 숫자 아닌 값이 있으면 날짜에서 보정
            bad_mask = ~df["조사년도"].astype(str).str.fullmatch(r"\d{4}", na=False)
            df.loc[bad_mask, "조사년도"] = df.loc[bad_mask, "조사일자(YYYYMMDD)"].dt.year
    else:
        # 날짜 컬럼이 없으면 해당 파일은 스킵하는 편이 안전
        print(f"[건너뜀] {os.path.basename(path)}: '조사일자(YYYYMMDD)' 컬럼 없음")
        return pd.DataFrame()

    # 2023–2024로 필터 후 비면 스킵
    if df.empty:
        print(f"[건너뜀] {os.path.basename(path)}: {YEAR_START}–{YEAR_END} 데이터 없음")
        return pd.DataFrame()

    # 병해 컬럼 선별
    pest_cols = [c for c in df.columns if any(k.casefold() in c.casefold() for k in PEST_KEYS)]
    if not pest_cols:
        print(f"[건너뜀] {os.path.basename(path)}: '{PEST_NAME}' 관련 컬럼 없음")
        return pd.DataFrame()

    # Core + 병해
    core_present = [c for c in CORE if c in df.columns]   # 실제 존재하는 Core만
    out = df[core_present + pest_cols].copy()

    # 병해값이 전부 결측인 행 제거(원치 않으면 주석 처리)
    out = out.dropna(subset=pest_cols, how="all")

    # Extras (옵션): Core/pest에 없는 나머지를 추가
    if KEEP_EXTRAS:
        used = set(core_present + pest_cols)
        extras = [c for c in df.columns if c not in used]
        if extras:
            if PREFIX_EXTRAS:
                crop = out["작물명"].iloc[0] if not out.empty else detect_crop_from_name(os.path.basename(path))
                extra_ren = {c: f"{crop}__{c}" for c in extras}
                out = pd.concat([out, df.loc[out.index, extras].rename(columns=extra_ren)], axis=1)
            else:
                out = pd.concat([out, df.loc[out.index, extras]], axis=1)

    # 정렬
    sort_keys = [c for c in ["지점ID", "조사일자(YYYYMMDD)"] if c in out.columns]
    if sort_keys:
        out = out.sort_values(sort_keys)

    return out


# === 메인: 파일 수집 ===
if RECURSIVE:
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "**", "RDA_OBSERVATION_*_with_siteid_dt.csv"), recursive=True))
else:
    files = sorted(glob.glob(os.path.join(INPUT_DIR, "RDA_OBSERVATION_*_with_siteid_dt.csv")))

if not files:
    raise SystemExit(f"대상 파일이 없습니다: {INPUT_DIR}")

# === 로드 & 병합 (작물별) ===
by_crop = {}
for fp in files:
    df_part = load_one(fp)
    if df_part.empty:
        continue
    crop = df_part["작물명"].iloc[0]
    by_crop.setdefault(crop, []).append(df_part)

if not by_crop:
    raise SystemExit("추출할 데이터가 없습니다. 경로/파일/키워드/연도 필터를 확인하세요.")

# === 저장 (작물별 CSV) ===
for crop, parts in by_crop.items():
    all_df = pd.concat(parts, ignore_index=True)

    # 안전 차원에서 한 번 더 날짜-연도 필터
    if "조사일자(YYYYMMDD)" in all_df.columns and pd.api.types.is_datetime64_any_dtype(all_df["조사일자(YYYYMMDD)"]):
        all_df = all_df[all_df["조사일자(YYYYMMDD)"].dt.year.between(YEAR_START, YEAR_END)]

    # 컬럼 재정렬: Core → 병해 → Extras
    core_present = [c for c in CORE if c in all_df.columns]
    pest_cols = [c for c in all_df.columns
                 if c not in core_present and any(k.casefold() in c.casefold() for k in PEST_KEYS)]
    extras = [c for c in all_df.columns if c not in core_present + pest_cols]
    all_df = all_df[core_present + pest_cols + extras]

    # 최종 정렬
    sort_keys = [c for c in ["지점ID", "조사일자(YYYYMMDD)"] if c in all_df.columns]
    if sort_keys:
        all_df = all_df.sort_values(sort_keys)

    save_all = os.path.join(OUT_DIR, f"{PEST_NAME}_{crop}_{YEAR_START}_{YEAR_END}.csv")
    all_df.to_csv(save_all, index=False, encoding="utf-8-sig")
    print(f"[저장] {save_all} (rows={len(all_df):,}, cols={len(all_df.columns)})")


### 기상데이터(2023-2024) + 복숭아심식나방

In [ ]:
#########23ㅇㅁㄹㅁㅇㅁㅇㅁㅇㅇㄹㅁㄹㅇㅁㄴㄹddfadfsfdafㅇㅈ랒ㅈㄹㄷㅈㄹ

### 복숭아심식나방 GDD

In [ ]:
# -*- coding: utf-8 -*-
"""
[두 해(2023, 2024) 전용] 10℃ 기준 누적온량(GDD10) + 첫 발생 요약
- 입력: 2023, 2024 파일만 처리
- 출력에 '연도' 컬럼 추가하지 않음
- 관측값: (트랩)복숭아심식나방-마리수
- 평균기온이 결측이면 (최고+최저)/2로 행 단위 대체
"""

import os, re
import numpy as np
import pandas as pd

# ================== 설정 ==================
INPUT_FILES = [
    "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples/joined_SAMPLE_2023_APPLE.csv",
    "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples/joined_SAMPLE_2024_APPLE.csv",
]
OUT_DIR    = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived"
BASE_TEMP  = 10.0  # 10℃ 기준
OBS_COL    = "(트랩)복숭아심식나방-마리수"
# =========================================

os.makedirs(OUT_DIR, exist_ok=True)

def tmean_series(df: pd.DataFrame) -> pd.Series:
    """평균기온 컬럼이 있으면 쓰되, 그 행이 NaN이면 (최고+최저)/2로 대체"""
    # 후보명(데이터셋에 따라 '℃'/'°C' 표기가 다를 수 있어 몇 가지 포함)
    mean_candidates = ["평균기온(°C)", "평균기온(℃)", "평균기온"]
    tmin_candidates = ["최저기온(°C)", "최저기온(℃)", "최저기온"]
    tmax_candidates = ["최고기온(°C)", "최고기온(℃)", "최고기온"]

    def pick_numeric(cands):
        for c in cands:
            if c in df.columns:
                return pd.to_numeric(df[c], errors="coerce")
        # 후보가 전혀 없으면 전부 NA 시리즈 반환
        return pd.Series(np.nan, index=df.index, dtype="float64")

    tmean = pick_numeric(mean_candidates)
    tmin  = pick_numeric(tmin_candidates)
    tmax  = pick_numeric(tmax_candidates)

    approx = (tmin + tmax) / 2.0
    # 평균기온이 NaN인 행만 (최고+최저)/2로 채움
    tmean = tmean.where(~tmean.isna(), approx)
    return tmean

def parse_year_crop_from_name(path: str):
    m = re.search(r"_(\d{4})_([A-Za-z]+)\.csv$", os.path.basename(path))
    year = int(m.group(1)) if m else None
    crop = m.group(2).upper() if m else "CROP"
    return year, crop

for INPUT_FILE in INPUT_FILES:
    YEAR_HINT, CROP = parse_year_crop_from_name(INPUT_FILE)

    # 1) 로드 (연도 컬럼 생성/출력 없음)
    df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig", parse_dates=["일시"])
    if not {"지점ID","일시"}.issubset(df.columns):
        raise SystemExit(f"[오류] 필수 컬럼('지점ID','일시')이 없습니다: {INPUT_FILE}")
    if OBS_COL not in df.columns:
        raise SystemExit(f"[오류] 관측 컬럼을 찾지 못했습니다: {OBS_COL}")

    # 파일명 연도만 필터(출력엔 '연도' 컬럼 없음)
    if YEAR_HINT is not None:
        df = df.loc[df["일시"].dt.year == YEAR_HINT].copy()
        if df.empty:
            raise SystemExit(f"[오류] {YEAR_HINT}년에 해당하는 데이터가 없습니다: {INPUT_FILE}")

    # 2) DD10 / GDD10 누적
    tmean = tmean_series(df)
    df["DD10"] = (tmean - BASE_TEMP).clip(lower=0)
    df = df.sort_values(["지점ID","일시"])
    df["GDD10_cum"] = df.groupby(["지점ID"], sort=False)["DD10"].cumsum()

    # 소수 1자리 반올림
    df["DD10"] = df["DD10"].round(1)
    df["GDD10_cum"] = df["GDD10_cum"].round(1)

    # 3) 관측값/발생여부 (트랩 마리수 기준)
    df["관측값"]   = pd.to_numeric(df[OBS_COL], errors="coerce")
    df["발생여부"] = (df["관측값"].fillna(0) > 0).astype(int)

    # 4) 첫 발생 요약 (연도 컬럼 추가 없음)
    has = df.loc[df["발생여부"]==1, ["지점ID","일시"]]
    first = has.groupby("지점ID", as_index=False).agg(첫발생일=("일시","min"))
    if not first.empty:
        gdd_at_first = (
            df.merge(first, on=["지점ID"], how="inner")
              .loc[lambda x: x["일시"].eq(x["첫발생일"]), ["지점ID","GDD10_cum"]]
              .rename(columns={"GDD10_cum":"첫발생_GDD10"})
        )
        first = first.merge(gdd_at_first, on="지점ID", how="left")
    else:
        first = pd.DataFrame(columns=["지점ID","첫발생일","첫발생_GDD10"])

    # 5) 저장
    year_tag = YEAR_HINT if YEAR_HINT is not None else "YEAR"
    daily_out = os.path.join(OUT_DIR, f"{CROP}_{year_tag}_GDD10_daily.csv")
    first_out = os.path.join(OUT_DIR, f"{CROP}_{year_tag}_GDD10_first_event.csv")

    df.to_csv(daily_out, index=False, encoding="utf-8-sig", float_format="%.1f")
    first.to_csv(first_out, index=False, encoding="utf-8-sig", float_format="%.1f")

    print("[저장]", daily_out,  "| rows=", len(df),    "| cols=", len(df.columns))
    print("[저장]", first_out, "| rows=", len(first), "| cols=", len(first.columns))


### 지점별 복숭아 심식나방 2023+2024 병합

In [ ]:
# -*- coding: utf-8 -*-
"""
GDD10 포함 일별 파일을 '연도 합본(2023+2024) → 지점ID별 분할'로 저장

입력: IN_DIR/{CROP}_{YEAR}_GDD10_daily.csv  (예: APPLE_2023_GDD10_daily.csv, APPLE_2024_GDD10_daily.csv)
처리: 같은 CROP의 2023/2024 파일을 먼저 concat하여 '합본'으로 만든 후, 지점ID별로 저장
출력: OUT_DIR/by_site/{CROP}/2023_2024/{CROP}_2023_2024_GDD10_daily_site-{지점ID}.csv
"""

import os, re, glob
import pandas as pd

# ===== 경로/옵션 =====
IN_DIR  = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived"
OUT_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived"

YEARS = {2023, 2024}   # 합칠 연도 세트
SKIP_NA_SITE = True    # 지점ID가 NaN이면 저장 스킵
DEDUPE = True          # (지점ID, 일시) 중복행 제거
SORT_BY_DATE = True    # 지점ID+일시 정렬

SITE_COL_PRIMARY  = "지점ID"
SITE_COL_FALLBACK = "지점"    # 일부 파일 호환용
DATE_COL = "일시"

# ============== 유틸 ==============
def parse_crop_year(fname: str):
    """
    파일명에서 CROP, YEAR 추출: {CROP}_{YEAR}_GDD10_daily.csv
    """
    m = re.search(r"([A-Za-z]+)_(\d{4})_GDD10_daily\.csv$", os.path.basename(fname))
    if not m:
        return None, None
    return m.group(1).upper(), int(m.group(2))

def sanitize_for_filename(val) -> str:
    s = "NA" if pd.isna(val) else str(val).strip()
    return re.sub(r'[^0-9A-Za-z가-힣_.-]+', "_", s)

# ============== 메인 ==============
files = sorted(glob.glob(os.path.join(IN_DIR, "*_GDD10_daily.csv")))
if not files:
    raise SystemExit(f"입력 파일을 찾지 못했습니다: {IN_DIR}/*_GDD10_daily.csv")

# 1) 파일을 CROP별로 묶고, 그 안에서 2023/2024만 고르기
by_crop = {}
for fp in files:
    crop, year = parse_crop_year(fp)
    if crop is None or year is None:
        continue
    if year not in YEARS:
        continue
    by_crop.setdefault(crop, []).append((year, fp))

if not by_crop:
    raise SystemExit("처리할 (2023/2024) 파일이 없습니다.")

for crop, lst in by_crop.items():
    # 연도 정렬 후 로드
    parts = []
    for year, path in sorted(lst, key=lambda x: x[0]):
        df = pd.read_csv(path, encoding="utf-8-sig")
        # 필수 컬럼
        site_col = SITE_COL_PRIMARY if SITE_COL_PRIMARY in df.columns else (
            SITE_COL_FALLBACK if SITE_COL_FALLBACK in df.columns else None
        )
        if site_col is None:
            raise SystemExit(f"[오류] '{SITE_COL_PRIMARY}'(또는 '{SITE_COL_FALLBACK}') 없음: {path}")
        if DATE_COL not in df.columns:
            raise SystemExit(f"[오류] '{DATE_COL}' 컬럼 없음: {path}")

        # 날짜 파싱 및 연도 필터(안전)
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce").dt.normalize()
        df = df[df[DATE_COL].dt.year.isin(YEARS)]
        # CROP/연도 정보는 파일명 기준이라 별도 컬럼 추가는 생략(원본 보존)

        parts.append(df)

    if not parts:
        print(f"[스킵] {crop}: 유효한 2023/2024 데이터 없음")
        continue

    # 2) 2023+2024 합본 만들기
    all_df = pd.concat(parts, ignore_index=True)

    # 중복 제거 / 정렬
    site_col = SITE_COL_PRIMARY if SITE_COL_PRIMARY in all_df.columns else (
        SITE_COL_FALLBACK if SITE_COL_FALLBACK in all_df.columns else None
    )
    if site_col is None:
        raise SystemExit(f"[오류] '{SITE_COL_PRIMARY}'(또는 '{SITE_COL_FALLBACK}') 컬럼 없음: {crop}")

    if DEDUPE:
        before = len(all_df)
        all_df = all_df.drop_duplicates(subset=[site_col, DATE_COL], keep="last")
        after = len(all_df)
        if before != after:
            print(f"[중복제거] {crop}: {before} → {after}")

    if SORT_BY_DATE:
        all_df = all_df.sort_values([site_col, DATE_COL])

    # 3) 지점별로 분할 저장 (연도 합본)
    span = "2023_2024"
    out_dir = os.path.join(OUT_DIR, "by_site", crop, span)
    os.makedirs(out_dir, exist_ok=True)

    total_files, total_rows = 0, 0
    for site, g in all_df.groupby(site_col, dropna=False):
        if SKIP_NA_SITE and pd.isna(site):
            continue
        site_str = sanitize_for_filename(site)
        out_path = os.path.join(out_dir, f"{crop}_{span}_GDD10_daily_site-{site_str}.csv")
        g.to_csv(out_path, index=False, encoding="utf-8-sig")
        total_files += 1
        total_rows  += len(g)
        print(f"[저장] {out_path} (rows={len(g):,}, cols={len(g.columns)})")

    print(f"[완료] {crop}: 합본 분할 저장 files={total_files:,}, rows={total_rows:,}, span={span}")


In [2]:
# -*- coding: utf-8 -*-
"""
지점별 2023+2024 합본 CSV(일별, GDD 포함) 1개를 열어
- 필수 컬럼 존재
- (지점ID, 일시) 중복
- 연도/기간 커버리지
- 핵심 피처 결측 현황
을 점검한다.
"""
import os, glob, re
import pandas as pd

# === 경로/대상 설정: 아래 2줄만 너 환경에 맞춰 확인 ===
CROP = "APPLE"
SITE_DIR = f"/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/{CROP}/2023_2024"
# 파일명 예: {CROP}_2023_2024_GDD10_daily_site-<지점ID>.csv

# (선택) 특정 지점ID를 직접 지정하고 싶으면 아래 주석 해제
TARGET_SITE = "38382_59884"

def pick_one_file():
    if 'TARGET_SITE' in globals():
        pat = os.path.join(SITE_DIR, f"{CROP}_2023_2024_GDD10_daily_site-{TARGET_SITE}.csv")
        files = glob.glob(pat)
    else:
        files = sorted(glob.glob(os.path.join(SITE_DIR, f"{CROP}_2023_2024_GDD10_daily_site-*.csv")))
    if not files:
        raise SystemExit(f"지점별 파일을 찾지 못했습니다: {SITE_DIR}")
    return files[0]

fp = pick_one_file()
print(f"[열기] {fp}")

df = pd.read_csv(fp, encoding="utf-8-sig")
# --- 날짜 파싱
if "일시" not in df.columns:
    raise SystemExit("필수 컬럼 누락: '일시'")
df["일시"] = pd.to_datetime(df["일시"], errors="coerce")
df["year"] = df["일시"].dt.year

# --- 필수/핵심 컬럼 체크
must_have = ["지점ID","일시","DD10","GDD10_cum"]
pest_pref = ["(트랩)복숭아심식나방-마리수","복숭아심식나방-피해가율(%)","복숭아심식나방-피해과수/750과"]
weather_any = [
    "평균기온(°C)","평균기온(℃)","최저기온(°C)","최저기온(℃)","최고기온(°C)","최고기온(℃)",
    "강수량(mm)","일강수량(mm)","합계일조시간(hr)","일조시간(hr)","수평면일사(MJ/m2)","일사량(MJ/m2)",
    "평균상대습도(%)","상대습도(%)","평균풍속(m/s)","평균풍속(㎧)"
]

missing = [c for c in must_have if c not in df.columns]
if missing:
    print(f"[경고] 필수 컬럼 누락: {missing}")

pest_found   = [c for c in pest_pref if c in df.columns]
weather_found= [c for c in weather_any if c in df.columns]

print("— 해충 컬럼(발견):", pest_found if pest_found else "없음")
print("— 기상 컬럼(일부):", weather_found[:6], ("…(+)" if len(weather_found)>6 else ""))

# --- (지점ID, 일시) 중복 검사
dup = df.duplicated(subset=["지점ID","일시"]).sum()
print(f"— 중복행(지점ID,일시 기준): {dup}")

# --- 연도/기간 요약
print("— 연도 분포:", df["year"].value_counts(dropna=True).to_dict())
print("— 날짜 범위:", df["일시"].min().date(), "→", df["일시"].max().date())

# --- 결측 현황(핵심만)
check_cols = pest_found + ["DD10","GDD10_cum"]
na_summary = {c: int(df[c].isna().sum()) for c in check_cols if c in df.columns}
print("— 결측 개수:", na_summary)

# --- 간단 샘플 확인
print(df.sort_values("일시").head(3)[["지점ID","일시"] + [c for c in ["(트랩)복숭아심식나방-마리수","DD10","GDD10_cum"] if c in df.columns]])


[열기] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024/APPLE_2023_2024_GDD10_daily_site-38382_59884.csv
— 해충 컬럼(발견): ['(트랩)복숭아심식나방-마리수', '복숭아심식나방-피해과수/750과']
— 기상 컬럼(일부): ['평균기온(°C)', '최저기온(°C)', '최고기온(°C)', '일강수량(mm)'] 
— 중복행(지점ID,일시 기준): 0
— 연도 분포: {2024: 366, 2023: 365}
— 날짜 범위: 2023-01-01 → 2024-12-31
— 결측 개수: {'(트랩)복숭아심식나방-마리수': 729, '복숭아심식나방-피해과수/750과': 729, 'DD10': 0, 'GDD10_cum': 0}
          지점ID         일시  (트랩)복숭아심식나방-마리수  DD10  GDD10_cum
0  38382_59884 2023-01-01              NaN   0.0        0.0
1  38382_59884 2023-01-02              NaN   0.0        0.0
2  38382_59884 2023-01-03              NaN   0.0        0.0


### 샘플  38206_60111

In [ ]:
# -*- coding: utf-8 -*-
"""
단일 지점(2023+2024 합본) → 30일 창 요약(오프셋 6종, 5일 간격)
- 2023: 연도 경계를 넘어도 30일씩 정확히 12창 생성 (cross)
- 2024: 연도 경계를 넘지 않음 (bounded) → 오프셋에 따라 11~12창
- 해충 칼럼 자동탐지(키워드), 접두사(pfm/ofm 등) 설정 가능
- 트랩 NA 정책: tails(첫 이전/마지막 이후 0), all(모든 NA=0), none(그대로)
- 저장 파일에는 cov_*, qc_* 제거
"""

import os
import re
import numpy as np
import pandas as pd

# =========[ 0) CONFIG ]=========
SITE_FILE = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024/APPLE_2023_2024_GDD10_daily_site-38206_60111.csv"
OUT_DIR   = os.path.join(os.path.dirname(SITE_FILE), "MovingAverage")

YEARS        = [2023, 2024]               # 처리할 연도
OFFSETS      = [0, 5, 10, 15, 20, 25]     # 5일 간격 6세트
WINDOW_DAYS  = 30                         # 창 길이(30일)
ROUND_DIGITS = 3                          # 출력 반올림 자릿수

# ---- 해충(동적) 설정 ----
PEST_NAME   = "복숭아심식나방"              # 설명/로그용
PEST_PREFIX = "pfm"                       # 출력 접두사(pfm/ofm/…); 공백이면 종-중립
PEST_KEYS   = ["복숭아심식나방", "심식나방", "pfm"]  # 트랩 칼럼 자동 탐지 키워드(대소문자 무시)
TRAP_COL_EXPLICIT = None                  # 트랩 칼럼명이 확정이면 직접 지정, 아니면 자동탐지

# 트랩 NA 처리: "none"(NA 유지) | "tails"(첫 이전, 마지막 이후 0) | "all"(모든 NA 0)
TRAP_NA_POLICY = "tails"

# ---- 연도별 창 정책 ----
# "cross"  : 연초부터 30일씩 딱 12창(연도 경계 넘어감)
# "bounded": 연도 경계 넘지 않음(연말 걸리면 창 수 11이 될 수 있음)
YEAR_WINDOW_POLICY = {2023: "cross", 2024: "bounded"}

# ---- 칼럼 이름(기상/온량) ----
COL = {
    "date":      "일시",
    "site":      "지점ID",
    "precip":    "일강수량(mm)",
    "tmax":      "최고기온(°C)",
    "tmin":      "최저기온(°C)",
    "tavg":      "평균기온(°C)",
    "wind_mean": "평균 풍속(m/s)",
    "wind_gust": "최대 풍속(m/s)",
    "dd10":      "DD10",
    # trap 은 동적 탐지
}

# =========[ 1) IO & BASICS ]=========
def load_site(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath, encoding="utf-8-sig")
    if COL["date"] not in df.columns or COL["site"] not in df.columns:
        raise SystemExit("필수 컬럼('일시','지점ID')이 없습니다.")
    df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce").dt.normalize()
    df = df.sort_values(COL["date"]).reset_index(drop=True)
    return df

def get_site_id_from_filename(filepath: str) -> str:
    m = re.search(r"_site-(.+)\.csv$", os.path.basename(filepath))
    return m.group(1) if m else "SITE"

# =========[ 2) TRAP COL DETECTION ]=========
def detect_trap_col(df: pd.DataFrame, pest_keys, explicit=None) -> str:
    if explicit:
        if explicit in df.columns:
            return explicit
        raise SystemExit(f"[에러] 지정한 TRAP_COL_EXPLICIT를 찾지 못함: {explicit}")

    def score(col_name: str) -> int:
        s = col_name.casefold()
        sc = 0
        if any(k.casefold() in s for k in pest_keys):
            sc += 3
        for kw, w in [("트랩", 2), ("trap", 2), ("마리수", 3), ("마리", 2), ("count", 2)]:
            if kw in s:
                sc += w
        for bad in ["일시", "지점id", "지점", "평균기온", "최고기온", "최저기온", "dd10"]:
            if bad in s:
                sc -= 5
        return sc

    candidates = sorted(df.columns, key=lambda c: score(c), reverse=True)
    best = candidates[0] if candidates else None
    if not best or score(best) <= 0:
        raise SystemExit("[에러] 트랩(마리수) 컬럼 자동 탐지 실패. TRAP_COL_EXPLICIT를 지정하세요.")
    return best

# =========[ 3) PREPROCESS ]=========
def fill_tavg_rowwise(df: pd.DataFrame) -> pd.Series:
    tavg = pd.to_numeric(df[COL["tavg"]], errors="coerce")
    tmin = pd.to_numeric(df[COL["tmin"]], errors="coerce")
    tmax = pd.to_numeric(df[COL["tmax"]], errors="coerce")
    approx = (tmin + tmax) / 2.0
    return tavg.where(~tavg.isna(), approx)

def qc_basic(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if COL["precip"] in out.columns:
        p = pd.to_numeric(out[COL["precip"]], errors="coerce")
        out[COL["precip"]] = p.where(p >= 0, 0.0)
    for k in ["wind_mean", "wind_gust"]:
        if COL[k] in out.columns:
            w = pd.to_numeric(out[COL[k]], errors="coerce")
            out[COL[k]] = w.where(w >= 0, 0.0)
    if COL["tmin"] in out.columns and COL["tmax"] in out.columns:
        tmin = pd.to_numeric(out[COL["tmin"]], errors="coerce")
        tmax = pd.to_numeric(out[COL["tmax"]], errors="coerce")
        swap = (tmin.notna() & tmax.notna() & (tmin > tmax))
        out.loc[swap, [COL["tmin"], COL["tmax"]]] = out.loc[swap, [COL["tmax"], COL["tmin"]]].values
    return out

def dailyize_trap(df: pd.DataFrame, trap_col: str) -> pd.Series:
    """
    트랩 관측(누적)을 일일 포획률로 분배.
    - 관측 v_i는 '직전 관측 다음날 ~ 관측일' Δ일에 균등분배(rate=v_i/Δ)
    - 첫 관측 전/마지막 관측 후: TRAP_NA_POLICY 적용
    """
    s = pd.Series(index=df[COL["date"]], data=pd.to_numeric(df[trap_col], errors="coerce").values, dtype="float64")
    s = s.sort_index()
    out = pd.Series(np.nan, index=s.index, dtype="float64")
    prev = None
    for d, v in s.dropna().items():
        v = float(v)
        if prev is None:
            out.loc[d] = (out.loc[d] if pd.notna(out.loc[d]) else 0.0) + v
        else:
            delta = (d - prev).days
            if delta <= 0:
                out.loc[d] = (out.loc[d] if pd.notna(out.loc[d]) else 0.0) + v
            else:
                rate = v / delta
                span = pd.date_range(prev + pd.Timedelta(days=1), d, freq="D")
                out.loc[span] = rate
        prev = d

    # NA 처리 정책
    if TRAP_NA_POLICY == "all":
        out = out.fillna(0.0)
    elif TRAP_NA_POLICY == "tails":
        first = out.first_valid_index()
        last  = out.last_valid_index()
        if first is not None:
            out.loc[: first - pd.Timedelta(days=1)] = 0.0
        if last is not None:
            out.loc[last + pd.Timedelta(days=1):] = 0.0
    # "none": 그대로 둠
    out.name = "trap_rate_daily"
    return out

# =========[ 4) WINDOWS ]=========
def make_windows(year: int, offset: int, win_days: int, policy: str):
    """
    policy='cross'  : 연초+offset에서 30일씩 '정확히 12창'
    policy='bounded': 연도 경계를 넘지 않게 창 생성(연말 걸리면 중단)
    """
    start0 = pd.Timestamp(f"{year}-01-01") + pd.Timedelta(days=offset)
    if policy == "cross":
        windows = []
        for i in range(12):
            ws = start0 + pd.Timedelta(days=i * win_days)
            we = ws + pd.Timedelta(days=win_days - 1)
            windows.append((i + 1, ws, we))
        return windows
    else:
        end_y = pd.Timestamp(f"{year}-12-31")
        windows, idx = [], 1
        while True:
            ws = start0 + pd.Timedelta(days=(idx - 1) * win_days)
            we = ws + pd.Timedelta(days=win_days - 1)
            if ws.year != year or we > end_y:
                break
            windows.append((idx, ws, we))
            idx += 1
        return windows

# =========[ 5) AGGREGATION ]=========
def agg_one_window(df: pd.DataFrame,
                   trap_rate: pd.Series,
                   start: pd.Timestamp,
                   end: pd.Timestamp,
                   pest_prefix: str = "pfm") -> dict:
    mask = (df[COL["date"]] >= start) & (df[COL["date"]] <= end)
    sub = df.loc[mask]
    days = (end - start).days + 1

    def mean_of(col):  return pd.to_numeric(sub[col], errors="coerce").mean()
    def sum_of(col):   return pd.to_numeric(sub[col], errors="coerce").sum(min_count=1)
    def count_valid(col): return pd.to_numeric(sub[col], errors="coerce").notna().sum()
    def max_of(col):   return pd.to_numeric(sub[col], errors="coerce").max()

    out = {
        "start_date": start.date(),
        "end_date": end.date(),
        "days": days,
        # 기온 평균
        "tavg_mean_30d": mean_of(COL["tavg"]),
        "tmin_mean_30d": mean_of(COL["tmin"]),
        "tmax_mean_30d": mean_of(COL["tmax"]),
        "cov_tavg_days": count_valid(COL["tavg"]),
        "cov_tmin_days": count_valid(COL["tmin"]),
        "cov_tmax_days": count_valid(COL["tmax"]),
        # 풍속
        "wind_mean_30d": mean_of(COL["wind_mean"]),
        "cov_wind_days": count_valid(COL["wind_mean"]),
        "wind_gust_max_30d": max_of(COL["wind_gust"]),
        "cov_gust_days": count_valid(COL["wind_gust"]),
        # 강수/온량
        "precip_sum_30d": sum_of(COL["precip"]),
        "cov_precip_days": count_valid(COL["precip"]),
        "gdd10_sum_30d": sum_of(COL["dd10"]),
        "cov_gdd_days": count_valid(COL["dd10"]),
    }

    # 트랩(일일화 평균)
    span = pd.date_range(start, end, freq="D")
    ts = trap_rate.reindex(span)
    trap_mean_col = f"{pest_prefix}_trap_rate_mean_30d" if pest_prefix else "trap_rate_mean_30d"
    out[trap_mean_col] = ts.mean()
    out["cov_trap_days"] = ts.notna().sum()

    return out

# =========[ 6) RUN ]=========
def main():
    os.makedirs(OUT_DIR, exist_ok=True)
    df = load_site(SITE_FILE)

    # 필수(기상/온량) 존재 확인
    needed = [COL[c] for c in ["precip","tmax","tmin","tavg","wind_mean","wind_gust","dd10"]]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise SystemExit(f"필수 칼럼이 없습니다(기상/온량): {missing}")

    # 트랩 칼럼 탐지
    trap_col = detect_trap_col(df, PEST_KEYS, TRAP_COL_EXPLICIT)
    print(f"[INFO] '{PEST_NAME}' 트랩 컬럼: {trap_col}")

    # 전처리: QC + 평균기온 보정
    df = qc_basic(df)
    df[COL["tavg"]] = fill_tavg_rowwise(df)

    # 트랩 일일화(+ NA 정책)
    trap_rate = dailyize_trap(df, trap_col=trap_col)

    # 사이트 ID
    site_id = get_site_id_from_filename(SITE_FILE)

    # 창 집계
    rows = []
    for year in YEARS:
        policy = YEAR_WINDOW_POLICY.get(year, "bounded")
        for offset in OFFSETS:
            for w_idx, ws, we in make_windows(year, offset, WINDOW_DAYS, policy):
                out = agg_one_window(df, trap_rate, ws, we, pest_prefix=PEST_PREFIX)
                out.update({"site_id": site_id, "year": year, "offset_days": offset, "window_idx": w_idx})
                rows.append(out)

    summary = pd.DataFrame(rows)

    # 반올림
    num_cols = [c for c in summary.columns if c.endswith(("_mean_30d","_sum_30d")) or c.endswith("_max_30d")]
    summary[num_cols] = summary[num_cols].round(ROUND_DIGITS)

    # 제출용: cov_*, qc_* 제거
    present = summary.drop(columns=[c for c in summary.columns if c.startswith("cov_") or c.startswith("qc_")],
                           errors="ignore")

    # 저장
    save_name = f"site-{site_id}_{PEST_PREFIX}_MovingAverage.csv" if PEST_PREFIX else f"site-{site_id}_MovingAverage.csv"
    save_path = os.path.join(OUT_DIR, save_name)
    present.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {save_path} (rows={len(present):,}, cols={len(present.columns)})")

if __name__ == "__main__":
    main()


[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024/summaries/site-38206_60111_pfm_30d_5dstep_summary.csv (rows=140, cols=15)


### 모든 지점별 이동평균 데이터

In [2]:
# -*- coding: utf-8 -*-
"""
단일 지점(2023+2024 합본) → 30일 창 요약(오프셋 6종, 5일 간격)
- 2023: 연도 경계를 넘어도 30일씩 정확히 12창 생성 (cross)
- 2024: 연도 경계를 넘지 않음 (bounded) → 오프셋에 따라 11~12창
- 해충 칼럼 자동탐지(키워드), 접두사(pfm/ofm 등) 설정 가능
- 트랩 NA 정책: tails(첫 이전/마지막 이후 0), all(모든 NA=0), none(그대로)
- 저장 파일에는 cov_*, qc_* 제거
"""

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# =========[ 0) CONFIG ]=========
# 대상 폴더(네가 준 경로)
TARGET_DIR = r"/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024"

YEARS        = [2023, 2024]               # 처리할 연도
OFFSETS      = [0, 5, 10, 15, 20, 25]     # 5일 간격 6세트
WINDOW_DAYS  = 30                         # 창 길이(30일)
ROUND_DIGITS = 3                          # 출력 반올림 자릿수

# ---- 해충(동적) 설정 ----
PEST_NAME   = "복숭아심식나방"              # 설명/로그용
PEST_PREFIX = "pfm"                       # 출력 접두사(pfm/ofm/…); 공백이면 종-중립
PEST_KEYS   = ["복숭아심식나방", "심식나방", "pfm"]  # 트랩 칼럼 자동 탐지 키워드(대소문자 무시)
TRAP_COL_EXPLICIT = None                  # 트랩 칼럼명이 확정이면 직접 지정, 아니면 자동탐지

# 트랩 NA 처리: "none"(NA 유지) | "tails"(첫 이전, 마지막 이후 0) | "all"(모든 NA 0)
TRAP_NA_POLICY = "tails"

# ---- 연도별 창 정책 ----
# "cross"  : 연초부터 30일씩 딱 12창(연도 경계 넘어감)
# "bounded": 연도 경계를 넘지 않게 창 생성(연말 걸리면 창 수 11이 될 수 있음)
YEAR_WINDOW_POLICY = {2023: "cross", 2024: "bounded"}

# ---- 칼럼 이름(기상/온량) ----
COL = {
    "date":      "일시",
    "site":      "지점ID",
    "precip":    "일강수량(mm)",
    "tmax":      "최고기온(°C)",
    "tmin":      "최저기온(°C)",
    "tavg":      "평균기온(°C)",
    "wind_mean": "평균 풍속(m/s)",
    "wind_gust": "최대 풍속(m/s)",
    "dd10":      "DD10",
    # trap 은 동적 탐지
}

# =========[ 1) IO & BASICS ]=========
def load_site(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath, encoding="utf-8-sig")
    if COL["date"] not in df.columns or COL["site"] not in df.columns:
        raise SystemExit("필수 컬럼('일시','지점ID')이 없습니다.")
    df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce").dt.normalize()
    df = df.sort_values(COL["date"]).reset_index(drop=True)
    return df

def get_site_id_from_filename(filepath: str) -> str:
    m = re.search(r"_site-(.+)\.csv$", os.path.basename(filepath))
    return m.group(1) if m else "SITE"

# =========[ 2) TRAP COL DETECTION ]=========
def detect_trap_col(df: pd.DataFrame, pest_keys, explicit=None) -> str:
    if explicit:
        if explicit in df.columns:
            return explicit
        raise SystemExit(f"[에러] 지정한 TRAP_COL_EXPLICIT를 찾지 못함: {explicit}")

    def score(col_name: str) -> int:
        s = col_name.casefold()
        sc = 0
        if any(k.casefold() in s for k in pest_keys):
            sc += 3
        for kw, w in [("트랩", 2), ("trap", 2), ("마리수", 3), ("마리", 2), ("count", 2)]:
            if kw in s:
                sc += w
        for bad in ["일시", "지점id", "지점", "평균기온", "최고기온", "최저기온", "dd10"]:
            if bad in s:
                sc -= 5
        return sc

    candidates = sorted(df.columns, key=lambda c: score(c), reverse=True)
    best = candidates[0] if candidates else None
    if not best or score(best) <= 0:
        raise SystemExit("[에러] 트랩(마리수) 컬럼 자동 탐지 실패. TRAP_COL_EXPLICIT를 지정하세요.")
    return best

# =========[ 3) PREPROCESS ]=========
def fill_tavg_rowwise(df: pd.DataFrame) -> pd.Series:
    tavg = pd.to_numeric(df[COL["tavg"]], errors="coerce")
    tmin = pd.to_numeric(df[COL["tmin"]], errors="coerce")
    tmax = pd.to_numeric(df[COL["tmax"]], errors="coerce")
    approx = (tmin + tmax) / 2.0
    return tavg.where(~tavg.isna(), approx)

def qc_basic(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if COL["precip"] in out.columns:
        p = pd.to_numeric(out[COL["precip"]], errors="coerce")
        out[COL["precip"]] = p.where(p >= 0, 0.0)
    for k in ["wind_mean", "wind_gust"]:
        if COL[k] in out.columns:
            w = pd.to_numeric(out[COL[k]], errors="coerce")
            out[COL[k]] = w.where(w >= 0, 0.0)
    if COL["tmin"] in out.columns and COL["tmax"] in out.columns:
        tmin = pd.to_numeric(out[COL["tmin"]], errors="coerce")
        tmax = pd.to_numeric(out[COL["tmax"]], errors="coerce")
        swap = (tmin.notna() & tmax.notna() & (tmin > tmax))
        out.loc[swap, [COL["tmin"], COL["tmax"]]] = out.loc[swap, [COL["tmax"], COL["tmin"]]].values
    return out

def dailyize_trap(df: pd.DataFrame, trap_col: str) -> pd.Series:
    """
    트랩 관측(누적)을 일일 포획률로 분배.
    - 관측 v_i는 '직전 관측 다음날 ~ 관측일' Δ일에 균등분배(rate=v_i/Δ)
    - 첫 관측 전/마지막 관측 후: TRAP_NA_POLICY 적용
    - ★ 관측이 단 1건도 없으면(전부 NA) 전 기간 0으로 간주
    """
    s = pd.Series(index=df[COL["date"]],
                  data=pd.to_numeric(df[trap_col], errors="coerce").values,
                  dtype="float64").sort_index()

    out = pd.Series(np.nan, index=s.index, dtype="float64")
    prev = None
    for d, v in s.dropna().items():
        v = float(v)
        if prev is None:
            out.loc[d] = (out.loc[d] if pd.notna(out.loc[d]) else 0.0) + v
        else:
            delta = (d - prev).days
            if delta <= 0:
                out.loc[d] = (out.loc[d] if pd.notna(out.loc[d]) else 0.0) + v
            else:
                rate = v / delta
                span = pd.date_range(prev + pd.Timedelta(days=1), d, freq="D")
                out.loc[span] = rate
        prev = d

    # NA 처리 정책
    if TRAP_NA_POLICY == "all":
        out = out.fillna(0.0)
    elif TRAP_NA_POLICY == "tails":
        first = out.first_valid_index()
        last  = out.last_valid_index()
        if first is not None:
            out.loc[: first - pd.Timedelta(days=1)] = 0.0
        if last is not None:
            out.loc[last + pd.Timedelta(days=1):] = 0.0

    # ★ 전부 NA(관측 전무)면 전 기간 0
    if not out.notna().any():
        out[:] = 0.0

    out.name = "trap_rate_daily"
    return out

# =========[ 4) WINDOWS ]=========
def make_windows(year: int, offset: int, win_days: int, policy: str):
    """
    policy='cross'  : 연초+offset에서 30일씩 '정확히 12창'
    policy='bounded': 연도 경계를 넘지 않게 창 생성(연말 걸리면 중단)
    """
    start0 = pd.Timestamp(f"{year}-01-01") + pd.Timedelta(days=offset)
    if policy == "cross":
        windows = []
        for i in range(12):
            ws = start0 + pd.Timedelta(days=i * win_days)
            we = ws + pd.Timedelta(days=win_days - 1)
            windows.append((i + 1, ws, we))
        return windows
    else:
        end_y = pd.Timestamp(f"{year}-12-31")
        windows, idx = [], 1
        while True:
            ws = start0 + pd.Timedelta(days=(idx - 1) * win_days)
            we = ws + pd.Timedelta(days=win_days - 1)
            if ws.year != year or we > end_y:
                break
            windows.append((idx, ws, we))
            idx += 1
        return windows

# =========[ 5) AGGREGATION ]=========
def agg_one_window(df: pd.DataFrame,
                   trap_rate: pd.Series,
                   start: pd.Timestamp,
                   end: pd.Timestamp,
                   pest_prefix: str = "pfm") -> dict:
    mask = (df[COL["date"]] >= start) & (df[COL["date"]] <= end)
    sub = df.loc[mask]
    days = (end - start).days + 1

    def mean_of(col):    return pd.to_numeric(sub[col], errors="coerce").mean()
    def sum_of(col):     return pd.to_numeric(sub[col], errors="coerce").sum(min_count=1)
    def count_valid(col):return pd.to_numeric(sub[col], errors="coerce").notna().sum()
    def max_of(col):     return pd.to_numeric(sub[col], errors="coerce").max()

    out = {
        "start_date": start.date(),
        "end_date": end.date(),
        "days": days,
        # 기온 평균
        "tavg_mean_30d": mean_of(COL["tavg"]),
        "tmin_mean_30d": mean_of(COL["tmin"]),
        "tmax_mean_30d": mean_of(COL["tmax"]),
        "cov_tavg_days": count_valid(COL["tavg"]),
        "cov_tmin_days": count_valid(COL["tmin"]),
        "cov_tmax_days": count_valid(COL["tmax"]),
        # 풍속
        "wind_mean_30d": mean_of(COL["wind_mean"]),
        "cov_wind_days": count_valid(COL["wind_mean"]),
        "wind_gust_max_30d": max_of(COL["wind_gust"]),
        "cov_gust_days": count_valid(COL["wind_gust"]),
        # 강수/온량
        "precip_sum_30d": sum_of(COL["precip"]),
        "cov_precip_days": count_valid(COL["precip"]),
        "gdd10_sum_30d": sum_of(COL["dd10"]),
        "cov_gdd_days": count_valid(COL["dd10"]),
    }

    # 트랩(일일화 평균)
    span = pd.date_range(start, end, freq="D")
    ts = trap_rate.reindex(span)
    trap_mean_col = f"{pest_prefix}_trap_rate_mean_30d" if pest_prefix else "trap_rate_mean_30d"
    out[trap_mean_col] = ts.mean()
    out["cov_trap_days"] = ts.notna().sum()

    return out

# =========[ 6) RUN (단일 파일 처리) ]=========
def process_one(site_file: str):
    out_dir = os.path.join(os.path.dirname(site_file), "MovingAverage")
    os.makedirs(out_dir, exist_ok=True)

    df = load_site(site_file)

    # 필수(기상/온량) 존재 확인
    needed = [COL[c] for c in ["precip","tmax","tmin","tavg","wind_mean","wind_gust","dd10"]]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"[SKIP] 필수 칼럼 없음: {Path(site_file).name} -> {missing}")
        return

    # 트랩 컬럼 탐지 → 실패 시 전 기간 0 처리
    try:
        trap_col = detect_trap_col(df, PEST_KEYS, TRAP_COL_EXPLICIT)
        print(f"[INFO] '{PEST_NAME}' 트랩 컬럼: {trap_col} @ {Path(site_file).name}")
        trap_rate = dailyize_trap(df, trap_col=trap_col)
    except Exception as e:
        print(f"[WARN] 트랩 컬럼 미발견({Path(site_file).name}): {e} → 전 기간 0 처리")
        trap_rate = pd.Series(0.0, index=df[COL['date']], dtype="float64", name="trap_rate_daily")

    # 전처리: QC + 평균기온 보정
    df = qc_basic(df)
    df[COL["tavg"]] = fill_tavg_rowwise(df)

    # 사이트 ID
    site_id = get_site_id_from_filename(site_file)

    # 창 집계
    rows = []
    for year in YEARS:
        policy = YEAR_WINDOW_POLICY.get(year, "bounded")
        for offset in OFFSETS:
            for w_idx, ws, we in make_windows(year, offset, WINDOW_DAYS, policy):
                out = agg_one_window(df, trap_rate, ws, we, pest_prefix=PEST_PREFIX)
                out.update({"site_id": site_id, "year": year, "offset_days": offset, "window_idx": w_idx})
                rows.append(out)

    summary = pd.DataFrame(rows)

    # 반올림
    num_cols = [c for c in summary.columns if c.endswith(("_mean_30d","_sum_30d")) or c.endswith("_max_30d")]
    summary[num_cols] = summary[num_cols].round(ROUND_DIGITS)

    # 제출용: cov_*, qc_* 제거
    present = summary.drop(columns=[c for c in summary.columns if c.startswith("cov_") or c.startswith("qc_")],
                           errors="ignore")

    # 저장
    save_name = f"site-{site_id}_{PEST_PREFIX}_MovingAverage.csv" if PEST_PREFIX else f"site-{site_id}_MovingAverage.csv"
    save_path = os.path.join(out_dir, save_name)
    present.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {save_path} (rows={len(present):,}, cols={len(present.columns)})")

# =========[ 7) BATCH: 폴더 내 모든 파일 처리 ]=========
def run_batch(target_dir: str,
              file_glob: str = "*_site-*.csv",
              recursive: bool = False):
    """
    target_dir 안의 입력 CSV들을 전부 처리.
    - file_glob: 입력 파일 패턴. (기본 *_site-*.csv)
    - recursive: 하위 폴더까지 처리할지 여부.
    """
    base = Path(target_dir)
    it = base.rglob(file_glob) if recursive else base.glob(file_glob)
    files = [p for p in it if p.is_file() and "MovingAverage" not in p.parts]

    if not files:
        print(f("[WARN] 처리할 파일이 없습니다: {target_dir} ({file_glob})"))
        return

    def natsort_key(s: str):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

    files = sorted(files, key=lambda p: natsort_key(str(p)))
    print(f"[INFO] 총 {len(files)}개 파일 처리 시작: {target_dir}")

    ok, fail = 0, 0
    for p in files:
        try:
            process_one(str(p))
            ok += 1
        except Exception as e:
            fail += 1
            print(f"[FAIL] {p.name}: {e}")

    print(f"[DONE] 성공 {ok} / 실패 {fail}")

# =========[ 8) ENTRY POINT ]=========
if __name__ == "__main__":
    # 폴더 전체 처리 (하위폴더는 기본 미포함)
    run_batch(TARGET_DIR, file_glob="*_site-*.csv", recursive=False)

    # 하위폴더까지 싹 돌리려면 아래 주석 해제:
    # run_batch(TARGET_DIR, file_glob="*_site-*.csv", recursive=True)


[INFO] 총 146개 파일 처리 시작: /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_GDD10_daily_site-30551_62994.csv
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024/MovingAverage/site-30551_62994_pfm_MovingAverage.csv (rows=140, cols=15)
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_GDD10_daily_site-30636_62943.csv
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024/MovingAverage/site-30636_62943_pfm_MovingAverage.csv (rows=140, cols=15)
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_GDD10_daily_site-30670_56490.csv
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_2024/MovingAverage/site-30670_56490_pfm_MovingAverage.csv (rows=140, cols=15)
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_GDD10_daily_site-30676_56412.csv
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site/APPLE/2023_202

### 여기부턴 2025 1월 데이터 붙히기 위한 작업

In [4]:
# sample_join_one.py
# -*- coding: utf-8 -*-
import os, glob
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

# ========== 설정 ==========
PEST_NAME = "복숭아심식나방"
PEST_KEYS = ["복숭아심식나방"]          # 해충 칼럼 자동 탐지 키워드(대/소문자 무관, 부분일치)
CROP = "APPLE"                          # "APPLE" / "BEAN" / "PEPPER" / "RICE"

# 생성할 연도(병해에 없어도 무조건 실행; 2025는 해충 칼럼이 NaN으로 유지됨)
YEARS_TO_USE = [2023, 2024, 2025]

WEATHER_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/ASOS+AWS"   # ASOS_AWS_YYYY.csv 위치
PEST_DIR    = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방"         # 복숭아심식나방_APPLE_2023_2024.csv 위치
OUT_DIR     = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples"
os.makedirs(OUT_DIR, exist_ok=True)
# ==========================

def read_csv_smart(path: str) -> pd.DataFrame:
    """인코딩을 몇 가지로 시도해 안전하게 읽기(윈도/맥 혼용 대비)."""
    for enc in ("utf-8-sig", "cp949", "euc-kr", "utf-8"):
        try:
            return pd.read_csv(path, encoding=enc, on_bad_lines="skip")
        except Exception:
            pass
    return pd.read_csv(path, on_bad_lines="skip")

def load_weather_year(y: int) -> pd.DataFrame:
    """연도별 기상 파일 로드 + 기본 정리(일시/지점 타입, 중복 제거)."""
    fp = os.path.join(WEATHER_DIR, f"ASOS_AWS_{y}.csv")
    if not os.path.exists(fp):
        raise FileNotFoundError(f"기상 연도 파일 없음: {fp}")
    w = read_csv_smart(fp)

    # 타입 정리
    if "일시" not in w.columns:
        raise ValueError(f"{fp}에 '일시' 컬럼이 없습니다.")
    w["일시"] = pd.to_datetime(w["일시"], errors="coerce").dt.normalize()

    for c in ("지점", "위도", "경도"):
        if c in w.columns:
            w[c] = pd.to_numeric(w[c], errors="coerce")

    # 중복 제거 & 정렬
    if {"지점", "일시"}.issubset(w.columns):
        w = w.drop_duplicates(subset=["지점", "일시"]).sort_values(["지점", "일시"])

    # 최근접 매핑용 관측소 좌표 확보 체크
    if not {"지점", "위도", "경도"}.issubset(w.columns):
        raise ValueError(f"{fp}에 '지점/위도/경도' 중 누락된 컬럼이 있습니다. (최근접 매핑 불가)")

    return w

def pest_cols(df: pd.DataFrame) -> list:
    """해충 칼럼 자동 탐지(키워드 부분일치, 대/소문자 무시)."""
    keys = [k.casefold() for k in PEST_KEYS]
    out = []
    for c in df.columns:
        cc = c.casefold()
        if any(k in cc for k in keys):
            out.append(c)
    return out

# 1) 병해(작물별) 로드
p_path = os.path.join(PEST_DIR, f"{PEST_NAME}_{CROP}_2023_2024.csv")
if not os.path.exists(p_path):
    raise SystemExit(f"병해 파일 없음: {p_path}")

p_all = read_csv_smart(p_path)

# 필수 컬럼 점검
need_cols = {"지점ID", "좌표-위도", "좌표-경도", "조사일자(YYYYMMDD)"}
missing = need_cols - set(p_all.columns)
if missing:
    raise SystemExit(f"병해 파일에 필요한 컬럼 없음: {missing}")

# 타입/일시 정리
p_all["일시"] = pd.to_datetime(p_all["조사일자(YYYYMMDD)"], errors="coerce").dt.normalize()
for c in ("좌표-위도", "좌표-경도"):
    p_all[c] = pd.to_numeric(p_all[c], errors="coerce")

# 해충 칼럼 목록
pcols = pest_cols(p_all)
if not pcols:
    print("⚠️ 해충 칼럼을 자동으로 찾지 못했습니다. PEST_KEYS를 확인하세요.")

# 2) 연도 루프 (병해에 없어도 지정 연도는 모두 생성)
years_to_use = YEARS_TO_USE

for y in years_to_use:
    print(f"\n=== {y}년 처리 시작 ===")
    # 2-1) 기상 로드 & 관측소 목록
    w = load_weather_year(y)
    stn = w[["지점", "위도", "경도"]].dropna().drop_duplicates()
    if stn.empty:
        raise SystemExit(f"{y}년 기상 데이터에서 관측소 좌표가 비어 있습니다.")

    # 2-2) 해충 지점ID -> 최근접 관측소(지점) 매핑표 만들기 (해충 파일의 모든 지점 사용)
    site_xy = (
        p_all[["지점ID", "좌표-위도", "좌표-경도"]]
        .dropna()
        .drop_duplicates()
        .copy()
    )
    if site_xy.empty:
        raise SystemExit("병해 파일에서 '지점ID/좌표-위도/좌표-경도'가 비어 있습니다.")

    tree = cKDTree(stn[["위도", "경도"]].to_numpy())
    dist, idx = tree.query(site_xy[["좌표-위도", "좌표-경도"]].to_numpy(), k=1)
    site_map = site_xy.assign(지점=stn.iloc[idx]["지점"].to_numpy())[["지점ID", "지점"]]

    # 2-3) 베이스 그리드 = (해충 파일의 모든 지점ID) × (연중 모든 날짜)
    days = pd.DataFrame({"일시": pd.date_range(f"{y}-01-01", f"{y}-12-31", freq="D")})
    base = site_map.assign(key=1).merge(days.assign(key=1), on="key").drop(columns="key")

    # 2-4) 기상 LEFT 조인 (관측소 지점 × 날짜로 붙음; 같은 지점에 매핑된 여러 지점ID도 각각 기상 복제되어 유지)
    base = base.merge(w, on=["지점", "일시"], how="left")

    # 2-5) 연도별 해충 테이블 준비
    use_cols = ["지점ID", "일시"] + pcols
    p_y = p_all.loc[p_all["일시"].dt.year == y, use_cols] if pcols else pd.DataFrame(columns=use_cols)

    # 해당 연도에 해충 값이 없으면 '컬럼만 있는 빈 DF'로 만들어서 컬럼 유지(→ 2025는 NaN 유지)
    if p_y.empty:
        p_y = pd.DataFrame(columns=use_cols)
    else:
        # 해충 컬럼이 전부 결측인 행은 제거(원치 않으면 이 줄을 주석 처리)
        p_y = p_y.dropna(subset=pcols, how="all")

    # 2-6) 해충 LEFT 조인 (키: 지점ID, 일시)
    out = base.merge(p_y, on=["지점ID", "일시"], how="left")

    # 2-7) 저장 전 정리(원하면 위경도/지점명 제거)
    drop_candidates = [c for c in ["지점명", "위도", "경도"] if c in out.columns]
    out = (
        out.drop(columns=drop_candidates)
           .sort_values(["지점ID", "일시"])
           .reset_index(drop=True)
    )

    # 2-8) 저장
    save_path = os.path.join(OUT_DIR, f"joined_SAMPLE2_{y}_{CROP}.csv")
    out.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {save_path}  rows={len(out):,}, cols={len(out.columns)}  (해충컬럼 유지 OK)")
    print(f"  · 해충 지점 수: {site_map['지점ID'].nunique()}, 관측소 수: {stn['지점'].nunique()}")

print("\n✅ 모든 연도 처리 완료!")



=== 2023년 처리 시작 ===
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples/joined_SAMPLE2_2023_APPLE.csv  rows=55,116, cols=12  (해충컬럼 유지 OK)
  · 해충 지점 수: 146, 관측소 수: 631

=== 2024년 처리 시작 ===
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples/joined_SAMPLE2_2024_APPLE.csv  rows=55,386, cols=12  (해충컬럼 유지 OK)
  · 해충 지점 수: 146, 관측소 수: 645

=== 2025년 처리 시작 ===
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples/joined_SAMPLE2_2025_APPLE.csv  rows=55,115, cols=12  (해충컬럼 유지 OK)
  · 해충 지점 수: 146, 관측소 수: 643

✅ 모든 연도 처리 완료!


In [5]:
import pandas as pd
src = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples/joined_SAMPLE2_2025_APPLE.csv"

df = pd.read_csv(src, encoding="utf-8-sig")
df["일시"] = pd.to_datetime(df["일시"], errors="coerce")
jan = df[df["일시"].dt.month == 1].copy()

# 덮어쓰기(원본을 1월 데이터만으로 교체)
jan.to_csv(src, index=False, encoding="utf-8-sig")
print(len(jan))

4681


In [6]:
# -*- coding: utf-8 -*-
"""
[2023, 2024, 2025 지원] 10℃ 기준 일일/누적 GDD10 + 첫 발생 요약
- 2025년은 1월만 사용(필요시 MONTH_FILTERS에서 수정)
- 해충 관측 칼럼이 없거나 전부 결측이어도 '관측값','발생여부' 칼럼을 유지
- 평균기온 결측은 (최고+최저)/2로 행 단위 보정
"""

import os, re
import numpy as np
import pandas as pd

# ================== 설정 ==================
JOINED_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_joined_samples"
OUT_DIR    = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived"
YEARS      = [2023, 2024, 2025]
CROP       = "APPLE"
BASE_TEMP  = 10.0  # 10℃ 기준

# 해충 관측 칼럼명(파일에 존재하지 않아도 됨: 없으면 NaN으로 관측값 생성)
OBS_COL    = "(트랩)복숭아심식나방-마리수"

# 연도별 사용할 월(없으면 전체). 2025은 1월만.
MONTH_FILTERS = {2025: [1]}
# =========================================

os.makedirs(OUT_DIR, exist_ok=True)

def tmean_series(df: pd.DataFrame) -> pd.Series:
    """평균기온이 있으면 사용, NaN은 (최저+최고)/2로 보정."""
    mean_candidates = ["평균기온(°C)", "평균기온(℃)", "평균기온"]
    tmin_candidates = ["최저기온(°C)", "최저기온(℃)", "최저기온"]
    tmax_candidates = ["최고기온(°C)", "최고기온(℃)", "최고기온"]

    def pick_numeric(cands):
        for c in cands:
            if c in df.columns:
                return pd.to_numeric(df[c], errors="coerce")
        return pd.Series(np.nan, index=df.index, dtype="float64")

    tmean = pick_numeric(mean_candidates)
    tmin  = pick_numeric(tmin_candidates)
    tmax  = pick_numeric(tmax_candidates)

    approx = (tmin + tmax) / 2.0
    tmean = tmean.where(~tmean.isna(), approx)
    return tmean

def parse_year_from_name(path: str):
    m = re.search(r"_(\d{4})_[A-Za-z]+\.csv$", os.path.basename(path))
    return int(m.group(1)) if m else None

def find_joined_path(year: int, crop: str) -> str:
    """joined 파일명 패턴을 유연하게 탐색."""
    candidates = [
        os.path.join(JOINED_DIR, f"joined_SAMPLE2_{year}_{crop}.csv"),
        os.path.join(JOINED_DIR, f"joined_SAMPLE_{year}_{crop}.csv"),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    # 최후의 수단: 패턴 스캔
    import glob
    pats = glob.glob(os.path.join(JOINED_DIR, f"joined*_{year}_{crop}.csv"))
    if pats:
        return sorted(pats)[0]
    raise FileNotFoundError(f"{year}년 입력 파일을 찾지 못했습니다: {candidates[0]}")

for year in YEARS:
    in_path = find_joined_path(year, CROP)
    df = pd.read_csv(in_path, encoding="utf-8-sig", parse_dates=["일시"])

    # 필수 키 점검
    if not {"지점ID", "일시"}.issubset(df.columns):
        raise SystemExit(f"[오류] '지점ID' 또는 '일시'가 없습니다: {in_path}")

    # (안전) 파일명 연도에 해당하는 데이터만 사용
    year_hint = parse_year_from_name(in_path)
    if year_hint:
        df = df.loc[df["일시"].dt.year == year_hint].copy()

    # 2025: 1월만 사용(필요 시 MONTH_FILTERS 수정)
    months = MONTH_FILTERS.get(year)
    if months is not None:
        df = df[df["일시"].dt.month.isin(months)].copy()

    if df.empty:
        print(f"[경고] {year}년 필터 결과가 비어 저장을 건너뜁니다: {in_path}")
        continue

    # ===== DD10 / GDD10 계산 =====
    tmean = tmean_series(df)
    df["DD10"] = (tmean - BASE_TEMP).clip(lower=0)
    df = df.sort_values(["지점ID", "일시"])
    df["GDD10_cum"] = df.groupby("지점ID", sort=False)["DD10"].cumsum()

    # 소수 1자리 반올림
    df["DD10"] = df["DD10"].round(1)
    df["GDD10_cum"] = df["GDD10_cum"].round(1)

    # ===== 관측값/발생여부 =====
    if OBS_COL in df.columns:
        obs_series = pd.to_numeric(df[OBS_COL], errors="coerce")
    else:
        # 관측 칼럼이 없어도 칼럼을 유지(전부 NaN)
        obs_series = pd.Series(np.nan, index=df.index, name=OBS_COL)

    df["관측값"] = obs_series
    df["발생여부"] = (df["관측값"].fillna(0) > 0).astype(int)

    # ===== 첫 발생 요약 =====
    has = df.loc[df["발생여부"] == 1, ["지점ID", "일시"]]
    if not has.empty:
        first = has.groupby("지점ID", as_index=False).agg(첫발생일=("일시", "min"))
        gdd_at_first = (
            df.merge(first, on=["지점ID"], how="inner")
              .loc[lambda x: x["일시"].eq(x["첫발생일"]), ["지점ID", "GDD10_cum"]]
              .rename(columns={"GDD10_cum": "첫발생_GDD10"})
        )
        first = first.merge(gdd_at_first, on="지점ID", how="left")
    else:
        first = pd.DataFrame(columns=["지점ID", "첫발생일", "첫발생_GDD10"])

    # ===== 저장 =====
    daily_out = os.path.join(OUT_DIR, f"{CROP}_{year}_GDD10_daily.csv")
    first_out = os.path.join(OUT_DIR, f"{CROP}_{year}_GDD10_first_event.csv")

    df.to_csv(daily_out, index=False, encoding="utf-8-sig", float_format="%.1f")
    first.to_csv(first_out, index=False, encoding="utf-8-sig", float_format="%.1f")

    print("[저장]", daily_out,  "| rows=", len(df),    "| cols=", len(df.columns))
    print("[저장]", first_out, "| rows=", len(first), "| cols=", len(first.columns))

print("✅ 모든 연도 처리 완료!")


[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/APPLE_2023_GDD10_daily.csv | rows= 55116 | cols= 16
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/APPLE_2023_GDD10_first_event.csv | rows= 54 | cols= 3
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/APPLE_2024_GDD10_daily.csv | rows= 55386 | cols= 16
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/APPLE_2024_GDD10_first_event.csv | rows= 46 | cols= 3
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/APPLE_2025_GDD10_daily.csv | rows= 4681 | cols= 16
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/APPLE_2025_GDD10_first_event.csv | rows= 0 | cols= 3
✅ 모든 연도 처리 완료!


In [7]:
# -*- coding: utf-8 -*-
"""
GDD10 포함 일별 파일을 '연도 합본(2023+2024+2025) → 지점ID별 분할'로 저장

입력: IN_DIR/{CROP}_{YEAR}_GDD10_daily.csv
     (예: APPLE_2023_GDD10_daily.csv, APPLE_2024_GDD10_daily.csv, APPLE_2025_GDD10_daily.csv)
처리: 같은 CROP의 지정 연도(2023/2024/2025)를 concat하여 '합본'으로 만든 후, 지점ID별로 저장
출력: OUT_DIR/by_site2/{CROP}/{연도스팬}/{CROP}_{연도스팬}_GDD10_daily_site-{지점ID}.csv
      (이미 존재하면 -001, -002 … 를 붙여 덮어쓰기 방지)
"""

import os, re, glob
import pandas as pd

# ===== 경로/옵션 =====
IN_DIR  = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived"
OUT_DIR = "/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived"

YEARS = {2023, 2024, 2025}   # 합칠 연도 세트
SKIP_NA_SITE = True          # 지점ID가 NaN이면 저장 스킵
DEDUPE = True                # (지점ID, 일시) 중복행 제거
SORT_BY_DATE = True          # 지점ID+일시 정렬

SITE_COL_PRIMARY  = "지점ID"
SITE_COL_FALLBACK = "지점"   # 일부 파일 호환용
DATE_COL = "일시"

# ============== 유틸 ==============
def parse_crop_year(fname: str):
    """파일명에서 CROP, YEAR 추출: {CROP}_{YEAR}_GDD10_daily.csv"""
    m = re.search(r"([A-Za-z]+)_(\d{4})_GDD10_daily\.csv$", os.path.basename(fname))
    if not m:
        return None, None
    return m.group(1).upper(), int(m.group(2))

def sanitize_for_filename(val) -> str:
    s = "NA" if pd.isna(val) else str(val).strip()
    return re.sub(r'[^0-9A-Za-z가-힣_.-]+', "_", s)

def year_span_label(years) -> str:
    ys = sorted(set(int(y) for y in years))
    return "_".join(str(y) for y in ys)

def unique_path(path: str) -> str:
    """이미 존재하면 -001, -002 … 를 붙여 덮어쓰기 방지."""
    if not os.path.exists(path):
        return path
    root, ext = os.path.splitext(path)
    i = 1
    while True:
        cand = f"{root}-{i:03d}{ext}"
        if not os.path.exists(cand):
            return cand
        i += 1

# ============== 메인 ==============
files = sorted(glob.glob(os.path.join(IN_DIR, "*_GDD10_daily.csv")))
if not files:
    raise SystemExit(f"입력 파일을 찾지 못했습니다: {IN_DIR}/*_GDD10_daily.csv")

# 1) 파일을 CROP별로 묶고, 그 안에서 YEARS만 고르기
by_crop = {}
for fp in files:
    crop, year = parse_crop_year(fp)
    if crop is None or year is None:
        continue
    if year not in YEARS:
        continue
    by_crop.setdefault(crop, []).append((year, fp))

if not by_crop:
    raise SystemExit(f"처리할 ({sorted(YEARS)}) 파일이 없습니다.")

for crop, lst in by_crop.items():
    # 연도 정렬 후 로드
    parts = []
    present_years = []
    for year, path in sorted(lst, key=lambda x: x[0]):
        df = pd.read_csv(path, encoding="utf-8-sig")
        # 필수 컬럼
        site_col = SITE_COL_PRIMARY if SITE_COL_PRIMARY in df.columns else (
            SITE_COL_FALLBACK if SITE_COL_FALLBACK in df.columns else None
        )
        if site_col is None:
            raise SystemExit(f"[오류] '{SITE_COL_PRIMARY}'(또는 '{SITE_COL_FALLBACK}') 없음: {path}")
        if DATE_COL not in df.columns:
            raise SystemExit(f"[오류] '{DATE_COL}' 컬럼 없음: {path}")

        # 날짜 파싱 및 연도 필터(안전)
        df[DATE_COL] = pd.to_datetime(df[DATE_COL], errors="coerce").dt.normalize()
        df = df[df[DATE_COL].dt.year == year]  # 파일명 연도만 확실히 사용

        parts.append(df)
        present_years.append(year)

    if not parts:
        print(f"[스킵] {crop}: 유효한 {sorted(YEARS)} 데이터 없음")
        continue

    # 2) 합본 만들기
    all_df = pd.concat(parts, ignore_index=True)

    # 중복 제거 / 정렬
    site_col = SITE_COL_PRIMARY if SITE_COL_PRIMARY in all_df.columns else (
        SITE_COL_FALLBACK if SITE_COL_FALLBACK in all_df.columns else None
    )
    if site_col is None:
        raise SystemExit(f"[오류] '{SITE_COL_PRIMARY}'(또는 '{SITE_COL_FALLBACK}') 컬럼 없음: {crop}")

    if DEDUPE:
        before = len(all_df)
        all_df = all_df.drop_duplicates(subset=[site_col, DATE_COL], keep="last")
        after = len(all_df)
        if before != after:
            print(f"[중복제거] {crop}: {before} → {after}")

    if SORT_BY_DATE:
        all_df = all_df.sort_values([site_col, DATE_COL])

    # 3) 지점별로 분할 저장 (연도 합본)
    span = year_span_label(present_years)  # 예: "2023_2024_2025"
    out_dir = os.path.join(OUT_DIR, "by_site2", crop, span)  # ← by_site2 로 저장
    os.makedirs(out_dir, exist_ok=True)

    total_files, total_rows = 0, 0
    for site, g in all_df.groupby(site_col, dropna=False):
        if SKIP_NA_SITE and pd.isna(site):
            continue
        site_str = sanitize_for_filename(site)
        base_path = os.path.join(out_dir, f"{crop}_{span}_GDD10_daily_site-{site_str}.csv")
        out_path = unique_path(base_path)  # ← 덮어쓰기 방지
        g.to_csv(out_path, index=False, encoding="utf-8-sig")
        total_files += 1
        total_rows  += len(g)
        print(f"[저장] {out_path} (rows={len(g):,}, cols={len(g.columns)})")

    print(f"[완료] {crop}: 합본 분할 저장 files={total_files:,}, rows={total_rows:,}, span={span}, dir={out_dir}")


[중복제거] APPLE: 115183 → 111252
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/APPLE_2023_2024_2025_GDD10_daily_site-30551_62994.csv (rows=762, cols=16)
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/APPLE_2023_2024_2025_GDD10_daily_site-30636_62943.csv (rows=762, cols=16)
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/APPLE_2023_2024_2025_GDD10_daily_site-30670_56490.csv (rows=762, cols=16)
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/APPLE_2023_2024_2025_GDD10_daily_site-30676_56412.csv (rows=762, cols=16)
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/APPLE_2023_2024_2025_GDD10_daily_site-30697_56554.csv (rows=762, cols=16)
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/APPLE_2023_2024_2025_GDD10_daily_site-30953_61888.csv (rows=762, cols=16)
[저장] /User

In [8]:
# -*- coding: utf-8 -*-
"""
단일 지점(2023+2024 합본) → 30일 창 요약(오프셋 6종, 5일 간격)
- 2023: 연도 경계를 넘어도 30일씩 정확히 12창 생성 (cross)
- 2024: 연도 경계를 넘지 않음 (bounded) → 오프셋에 따라 11~12창
- 해충 칼럼 자동탐지(키워드), 접두사(pfm/ofm 등) 설정 가능
- 트랩 NA 정책: tails(첫 이전/마지막 이후 0), all(모든 NA=0), none(그대로)
- 저장 파일에는 cov_*, qc_* 제거
"""

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path

# =========[ 0) CONFIG ]=========
# 대상 폴더(네가 준 경로)
TARGET_DIR = r"/Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025"

YEARS        = [2023, 2024]               # 처리할 연도
OFFSETS      = [0, 5, 10, 15, 20, 25]     # 5일 간격 6세트
WINDOW_DAYS  = 30                         # 창 길이(30일)
ROUND_DIGITS = 3                          # 출력 반올림 자릿수

# ---- 해충(동적) 설정 ----
PEST_NAME   = "복숭아심식나방"              # 설명/로그용
PEST_PREFIX = "pfm"                       # 출력 접두사(pfm/ofm/…); 공백이면 종-중립
PEST_KEYS   = ["복숭아심식나방", "심식나방", "pfm"]  # 트랩 칼럼 자동 탐지 키워드(대소문자 무시)
TRAP_COL_EXPLICIT = None                  # 트랩 칼럼명이 확정이면 직접 지정, 아니면 자동탐지

# 트랩 NA 처리: "none"(NA 유지) | "tails"(첫 이전, 마지막 이후 0) | "all"(모든 NA 0)
TRAP_NA_POLICY = "tails"

# ---- 연도별 창 정책 ----
# "cross"  : 연초부터 30일씩 딱 12창(연도 경계 넘어감)
# "bounded": 연도 경계를 넘지 않게 창 생성(연말 걸리면 창 수 11이 될 수 있음)
YEAR_WINDOW_POLICY = {2023: "cross", 2024: "cross"}

# ---- 칼럼 이름(기상/온량) ----
COL = {
    "date":      "일시",
    "site":      "지점ID",
    "precip":    "일강수량(mm)",
    "tmax":      "최고기온(°C)",
    "tmin":      "최저기온(°C)",
    "tavg":      "평균기온(°C)",
    "wind_mean": "평균 풍속(m/s)",
    "wind_gust": "최대 풍속(m/s)",
    "dd10":      "DD10",
    # trap 은 동적 탐지
}

# =========[ 1) IO & BASICS ]=========
def load_site(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath, encoding="utf-8-sig")
    if COL["date"] not in df.columns or COL["site"] not in df.columns:
        raise SystemExit("필수 컬럼('일시','지점ID')이 없습니다.")
    df[COL["date"]] = pd.to_datetime(df[COL["date"]], errors="coerce").dt.normalize()
    df = df.sort_values(COL["date"]).reset_index(drop=True)
    return df

def get_site_id_from_filename(filepath: str) -> str:
    m = re.search(r"_site-(.+)\.csv$", os.path.basename(filepath))
    return m.group(1) if m else "SITE"

# =========[ 2) TRAP COL DETECTION ]=========
def detect_trap_col(df: pd.DataFrame, pest_keys, explicit=None) -> str:
    if explicit:
        if explicit in df.columns:
            return explicit
        raise SystemExit(f"[에러] 지정한 TRAP_COL_EXPLICIT를 찾지 못함: {explicit}")

    def score(col_name: str) -> int:
        s = col_name.casefold()
        sc = 0
        if any(k.casefold() in s for k in pest_keys):
            sc += 3
        for kw, w in [("트랩", 2), ("trap", 2), ("마리수", 3), ("마리", 2), ("count", 2)]:
            if kw in s:
                sc += w
        for bad in ["일시", "지점id", "지점", "평균기온", "최고기온", "최저기온", "dd10"]:
            if bad in s:
                sc -= 5
        return sc

    candidates = sorted(df.columns, key=lambda c: score(c), reverse=True)
    best = candidates[0] if candidates else None
    if not best or score(best) <= 0:
        raise SystemExit("[에러] 트랩(마리수) 컬럼 자동 탐지 실패. TRAP_COL_EXPLICIT를 지정하세요.")
    return best

# =========[ 3) PREPROCESS ]=========
def fill_tavg_rowwise(df: pd.DataFrame) -> pd.Series:
    tavg = pd.to_numeric(df[COL["tavg"]], errors="coerce")
    tmin = pd.to_numeric(df[COL["tmin"]], errors="coerce")
    tmax = pd.to_numeric(df[COL["tmax"]], errors="coerce")
    approx = (tmin + tmax) / 2.0
    return tavg.where(~tavg.isna(), approx)

def qc_basic(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if COL["precip"] in out.columns:
        p = pd.to_numeric(out[COL["precip"]], errors="coerce")
        out[COL["precip"]] = p.where(p >= 0, 0.0)
    for k in ["wind_mean", "wind_gust"]:
        if COL[k] in out.columns:
            w = pd.to_numeric(out[COL[k]], errors="coerce")
            out[COL[k]] = w.where(w >= 0, 0.0)
    if COL["tmin"] in out.columns and COL["tmax"] in out.columns:
        tmin = pd.to_numeric(out[COL["tmin"]], errors="coerce")
        tmax = pd.to_numeric(out[COL["tmax"]], errors="coerce")
        swap = (tmin.notna() & tmax.notna() & (tmin > tmax))
        out.loc[swap, [COL["tmin"], COL["tmax"]]] = out.loc[swap, [COL["tmax"], COL["tmin"]]].values
    return out

def dailyize_trap(df: pd.DataFrame, trap_col: str) -> pd.Series:
    """
    트랩 관측(누적)을 일일 포획률로 분배.
    - 관측 v_i는 '직전 관측 다음날 ~ 관측일' Δ일에 균등분배(rate=v_i/Δ)
    - 첫 관측 전/마지막 관측 후: TRAP_NA_POLICY 적용
    - ★ 관측이 단 1건도 없으면(전부 NA) 전 기간 0으로 간주
    """
    s = pd.Series(index=df[COL["date"]],
                  data=pd.to_numeric(df[trap_col], errors="coerce").values,
                  dtype="float64").sort_index()

    out = pd.Series(np.nan, index=s.index, dtype="float64")
    prev = None
    for d, v in s.dropna().items():
        v = float(v)
        if prev is None:
            out.loc[d] = (out.loc[d] if pd.notna(out.loc[d]) else 0.0) + v
        else:
            delta = (d - prev).days
            if delta <= 0:
                out.loc[d] = (out.loc[d] if pd.notna(out.loc[d]) else 0.0) + v
            else:
                rate = v / delta
                span = pd.date_range(prev + pd.Timedelta(days=1), d, freq="D")
                out.loc[span] = rate
        prev = d

    # NA 처리 정책
    if TRAP_NA_POLICY == "all":
        out = out.fillna(0.0)
    elif TRAP_NA_POLICY == "tails":
        first = out.first_valid_index()
        last  = out.last_valid_index()
        if first is not None:
            out.loc[: first - pd.Timedelta(days=1)] = 0.0
        if last is not None:
            out.loc[last + pd.Timedelta(days=1):] = 0.0

    # ★ 전부 NA(관측 전무)면 전 기간 0
    if not out.notna().any():
        out[:] = 0.0

    out.name = "trap_rate_daily"
    return out

# =========[ 4) WINDOWS ]=========
def make_windows(year: int, offset: int, win_days: int, policy: str):
    """
    policy='cross'  : 연초+offset에서 30일씩 '정확히 12창'
    policy='bounded': 연도 경계를 넘지 않게 창 생성(연말 걸리면 중단)
    """
    start0 = pd.Timestamp(f"{year}-01-01") + pd.Timedelta(days=offset)
    if policy == "cross":
        windows = []
        for i in range(12):
            ws = start0 + pd.Timedelta(days=i * win_days)
            we = ws + pd.Timedelta(days=win_days - 1)
            windows.append((i + 1, ws, we))
        return windows
    else:
        end_y = pd.Timestamp(f"{year}-12-31")
        windows, idx = [], 1
        while True:
            ws = start0 + pd.Timedelta(days=(idx - 1) * win_days)
            we = ws + pd.Timedelta(days=win_days - 1)
            if ws.year != year or we > end_y:
                break
            windows.append((idx, ws, we))
            idx += 1
        return windows

# =========[ 5) AGGREGATION ]=========
def agg_one_window(df: pd.DataFrame,
                   trap_rate: pd.Series,
                   start: pd.Timestamp,
                   end: pd.Timestamp,
                   pest_prefix: str = "pfm") -> dict:
    mask = (df[COL["date"]] >= start) & (df[COL["date"]] <= end)
    sub = df.loc[mask]
    days = (end - start).days + 1

    def mean_of(col):    return pd.to_numeric(sub[col], errors="coerce").mean()
    def sum_of(col):     return pd.to_numeric(sub[col], errors="coerce").sum(min_count=1)
    def count_valid(col):return pd.to_numeric(sub[col], errors="coerce").notna().sum()
    def max_of(col):     return pd.to_numeric(sub[col], errors="coerce").max()

    out = {
        "start_date": start.date(),
        "end_date": end.date(),
        "days": days,
        # 기온 평균
        "tavg_mean_30d": mean_of(COL["tavg"]),
        "tmin_mean_30d": mean_of(COL["tmin"]),
        "tmax_mean_30d": mean_of(COL["tmax"]),
        "cov_tavg_days": count_valid(COL["tavg"]),
        "cov_tmin_days": count_valid(COL["tmin"]),
        "cov_tmax_days": count_valid(COL["tmax"]),
        # 풍속
        "wind_mean_30d": mean_of(COL["wind_mean"]),
        "cov_wind_days": count_valid(COL["wind_mean"]),
        "wind_gust_max_30d": max_of(COL["wind_gust"]),
        "cov_gust_days": count_valid(COL["wind_gust"]),
        # 강수/온량
        "precip_sum_30d": sum_of(COL["precip"]),
        "cov_precip_days": count_valid(COL["precip"]),
        "gdd10_sum_30d": sum_of(COL["dd10"]),
        "cov_gdd_days": count_valid(COL["dd10"]),
    }

    # 트랩(일일화 평균)
    span = pd.date_range(start, end, freq="D")
    ts = trap_rate.reindex(span)
    trap_mean_col = f"{pest_prefix}_trap_rate_mean_30d" if pest_prefix else "trap_rate_mean_30d"
    out[trap_mean_col] = ts.mean()
    out["cov_trap_days"] = ts.notna().sum()

    return out

# =========[ 6) RUN (단일 파일 처리) ]=========
def process_one(site_file: str):
    out_dir = os.path.join(os.path.dirname(site_file), "MovingAverage")
    os.makedirs(out_dir, exist_ok=True)

    df = load_site(site_file)

    # 필수(기상/온량) 존재 확인
    needed = [COL[c] for c in ["precip","tmax","tmin","tavg","wind_mean","wind_gust","dd10"]]
    missing = [c for c in needed if c not in df.columns]
    if missing:
        print(f"[SKIP] 필수 칼럼 없음: {Path(site_file).name} -> {missing}")
        return

    # 트랩 컬럼 탐지 → 실패 시 전 기간 0 처리
    try:
        trap_col = detect_trap_col(df, PEST_KEYS, TRAP_COL_EXPLICIT)
        print(f"[INFO] '{PEST_NAME}' 트랩 컬럼: {trap_col} @ {Path(site_file).name}")
        trap_rate = dailyize_trap(df, trap_col=trap_col)
    except Exception as e:
        print(f"[WARN] 트랩 컬럼 미발견({Path(site_file).name}): {e} → 전 기간 0 처리")
        trap_rate = pd.Series(0.0, index=df[COL['date']], dtype="float64", name="trap_rate_daily")

    # 전처리: QC + 평균기온 보정
    df = qc_basic(df)
    df[COL["tavg"]] = fill_tavg_rowwise(df)

    # 사이트 ID
    site_id = get_site_id_from_filename(site_file)

    # 창 집계
    rows = []
    for year in YEARS:
        policy = YEAR_WINDOW_POLICY.get(year, "bounded")
        for offset in OFFSETS:
            for w_idx, ws, we in make_windows(year, offset, WINDOW_DAYS, policy):
                out = agg_one_window(df, trap_rate, ws, we, pest_prefix=PEST_PREFIX)
                out.update({"site_id": site_id, "year": year, "offset_days": offset, "window_idx": w_idx})
                rows.append(out)

    summary = pd.DataFrame(rows)

    # 반올림
    num_cols = [c for c in summary.columns if c.endswith(("_mean_30d","_sum_30d")) or c.endswith("_max_30d")]
    summary[num_cols] = summary[num_cols].round(ROUND_DIGITS)

    # 제출용: cov_*, qc_* 제거
    present = summary.drop(columns=[c for c in summary.columns if c.startswith("cov_") or c.startswith("qc_")],
                           errors="ignore")

    # 저장
    save_name = f"site-{site_id}_{PEST_PREFIX}_MovingAverage.csv" if PEST_PREFIX else f"site-{site_id}_MovingAverage.csv"
    save_path = os.path.join(out_dir, save_name)
    present.to_csv(save_path, index=False, encoding="utf-8-sig")
    print(f"[저장] {save_path} (rows={len(present):,}, cols={len(present.columns)})")

# =========[ 7) BATCH: 폴더 내 모든 파일 처리 ]=========
def run_batch(target_dir: str,
              file_glob: str = "*_site-*.csv",
              recursive: bool = False):
    """
    target_dir 안의 입력 CSV들을 전부 처리.
    - file_glob: 입력 파일 패턴. (기본 *_site-*.csv)
    - recursive: 하위 폴더까지 처리할지 여부.
    """
    base = Path(target_dir)
    it = base.rglob(file_glob) if recursive else base.glob(file_glob)
    files = [p for p in it if p.is_file() and "MovingAverage" not in p.parts]

    if not files:
        print(f("[WARN] 처리할 파일이 없습니다: {target_dir} ({file_glob})"))
        return

    def natsort_key(s: str):
        return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", s)]

    files = sorted(files, key=lambda p: natsort_key(str(p)))
    print(f"[INFO] 총 {len(files)}개 파일 처리 시작: {target_dir}")

    ok, fail = 0, 0
    for p in files:
        try:
            process_one(str(p))
            ok += 1
        except Exception as e:
            fail += 1
            print(f"[FAIL] {p.name}: {e}")

    print(f"[DONE] 성공 {ok} / 실패 {fail}")

# =========[ 8) ENTRY POINT ]=========
if __name__ == "__main__":
    # 폴더 전체 처리 (하위폴더는 기본 미포함)
    run_batch(TARGET_DIR, file_glob="*_site-*.csv", recursive=False)

    # 하위폴더까지 싹 돌리려면 아래 주석 해제:
    # run_batch(TARGET_DIR, file_glob="*_site-*.csv", recursive=True)


[INFO] 총 146개 파일 처리 시작: /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_2025_GDD10_daily_site-30551_62994.csv
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/MovingAverage/site-30551_62994_pfm_MovingAverage.csv (rows=144, cols=15)
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_2025_GDD10_daily_site-30636_62943.csv
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/MovingAverage/site-30636_62943_pfm_MovingAverage.csv (rows=144, cols=15)
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_2025_GDD10_daily_site-30670_56490.csv
[저장] /Users/doyoung-gil/Desktop/연구실/데이터/복숭아심식나방_derived/by_site2/APPLE/2023_2024_2025/MovingAverage/site-30670_56490_pfm_MovingAverage.csv (rows=144, cols=15)
[INFO] '복숭아심식나방' 트랩 컬럼: (트랩)복숭아심식나방-마리수 @ APPLE_2023_2024_2025_GDD10_daily_site-30676_56412.csv
[저장] /Users/doyoung-gil/Desktop/연구